# HeatReady walkthrough: New York City by census tract

This walkthrough repeats the analyses from the HeatReady presentation with the Python or R client. It
starts with one hot day across Manhattan and Brooklyn, finds the census tracts where nights stayed hot
through the summer, and then looks at where adults 75 and older and children under 5 live within those
tracts. The last steps bring in outside data, create a project from your own boundaries, and run the same map
for any other public project.

Every step runs against the live HeatReady API. The code comes in Python and R. Pick one language and
follow it through; the results are the same.

## Before you start

You need a HeatReady username and key. At a workshop, claim one at
[nishantkishore.com/workshop](https://nishantkishore.com/workshop) with the code from the slide.
Otherwise, write to [datascience_crisisready@harvard.edu](mailto:datascience_crisisready@harvard.edu).

Install the client and the packages this walkthrough uses.

```bash
# Python 3.10 or newer
pip install "git+https://github.com/crisisready/heat-ready-clients.git#subdirectory=python" geopandas matplotlib
```

The code reads your username and key from two environment variables, `HEATREADY_USERNAME` and
`HEATREADY_KEY`. The workshop page shows the exact lines to set them in a terminal, in Python, or in R.
Keep the key out of any file you plan to share.

## 1. Connect

`ping()` checks that your username and key work. The project for this walkthrough is
`nyc-manhattan-brooklyn-2026`, the 1,114 census tracts of Manhattan and Brooklyn. It is public, so any
HeatReady account can read it.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from heatready import HeatReadyClient

client = HeatReadyClient(
    username=os.environ["HEATREADY_USERNAME"],
    key=os.environ["HEATREADY_KEY"],
)
print(client.ping())

PROJECT = "nyc-manhattan-brooklyn-2026"
status = client.get_project_status(PROJECT)
print(status["polygon_count"], "census tracts")

The API does not return tract boundaries, so the shapes for mapping come from a file on the workshop
site. Each tract's `name` matches the name the API uses, and `neighborhood` is its New York City
Neighborhood Tabulation Area.

In [ ]:
TRACTS_URL = "https://nishantkishore.com/workshop/files/nyc-tracts.geojson"
tracts = gpd.read_file(TRACTS_URL)
tracts.plot(color="lightgrey", edgecolor="white", linewidth=0.2, figsize=(6, 7)).set_axis_off()

## 2. One hot day, tract by tract

Each row from `iter_metrics()` is one tract on one local day. We pull 5 June 2026, the day the
presentation opens on.

Two values matter here. `day_t2m_max` is the daily high from the ERA5-Land grid, whose cells are about
9 km across. The `downscaled` block carries the same high after the HeatReady model adjusts it for the
tract's own surroundings, such as land cover, tree canopy, and elevation. Where a tract has no
adjustment, we keep the grid value.

In [ ]:
rows = list(client.iter_metrics(PROJECT, date_from="2026-06-05", date_to="2026-06-05"))

def tract_high(row):
    ds = row.get("downscaled") or {}
    return ds.get("metrics", {}).get("day_t2m_max", row["day_t2m_max"])

day = pd.DataFrame({
    "name": [r["name"] for r in rows],
    "grid_high": [r["day_t2m_max"] for r in rows],
    "tract_high": [tract_high(r) for r in rows],
})
print(day["grid_high"].round(2).nunique(), "distinct grid values")
print(day[["grid_high", "tract_high"]].quantile([0.05, 0.95]).round(1))

The grid gives the two boroughs 6 different values, spread over 1.5 °C. Across the tracts, the high
runs from 32.4 to 34.3 °C between the 5th and 95th percentiles. The map below puts every tract in
fixed half-degree classes, so one colour always means the same temperature.

In [ ]:
BREAKS = [-np.inf, 32, 32.5, 33, 33.5, 34, 34.5, 35, np.inf]
LABELS = ["below 32", "32 to 32.5", "32.5 to 33", "33 to 33.5", "33.5 to 34", "34 to 34.5", "34.5 to 35", "35 and above"]
COLOURS = ["#4575b4", "#74add1", "#abd9e9", "#e0f3f8", "#fee090", "#fdae61", "#f46d43", "#d73027"]

hot_day = tracts.merge(day, on="name")
hot_day["class"] = pd.cut(hot_day["tract_high"], BREAKS, labels=LABELS)
ax = hot_day.plot(column="class", cmap=ListedColormap(COLOURS), legend=True, figsize=(7, 8),
                  legend_kwds={"title": "Daily high (°C)", "loc": "upper left", "bbox_to_anchor": (1, 1)})
ax.set_title("Daily high, 5 June 2026")
ax.set_axis_off()

print(hot_day.nlargest(5, "tract_high")[["neighborhood", "borough", "tract_high"]])

The hottest tracts that afternoon were in Inwood and East Harlem, up to 2.6 °C above the grid cell
they sit in.

## 3. The highest-risk tracts: nights that stayed hot

The body recovers from a hot day after dark, so a night that stays warm carries the risk forward.
`get_nighttime_persistence()` counts, for each tract, the nights this season when the temperature
never fell below 20 °C. We call the top quarter of tracts on that count the highest-risk tracts, the
same cut the presentation uses.

In [ ]:
nights = client.get_nighttime_persistence(PROJECT)
hot_nights = pd.DataFrame({
    "name": list(nights["per_tract"]),
    "hot_nights": [t["no_relief_count_season"] for t in nights["per_tract"].values()],
    "nights_tracked": [t["n_nights_tracked"] for t in nights["per_tract"].values()],
})

risk = tracts.merge(hot_nights, on="name")
cutoff = risk["hot_nights"].quantile(0.75)
risk["highest_risk"] = risk["hot_nights"] >= cutoff
print("As of", nights["as_of_date"], "- cutoff:", cutoff, "hot nights")
print(risk["highest_risk"].sum(), "highest-risk tracts")
print(risk[risk["highest_risk"]]["neighborhood"].value_counts().head(6))

ax = risk.plot(color="lightgrey", figsize=(7, 8))
risk[risk["highest_risk"]].plot(ax=ax, color="#d73027")
ax.set_title(f"Top quarter of tracts for hot nights (at least {cutoff:.0f})")
ax.set_axis_off()

On 28 September 2026, 311 tracts had at least 74 hot nights out of the 120 tracked since 1 June.
Many tracts share the same count, so the top quarter holds a few more than a quarter of the city. 284
of the 311 are in Brooklyn, with the most in Borough Park, Bensonhurst, and Brownsville. The count
updates every day, so a later run gives slightly different numbers.

## 4. Who lives in the highest-risk tracts

`get_vulnerability_data()` returns population by age group for each tract, from WorldPop. It
returns 500 tracts per call unless asked for more, so we set `limit` above the project's 1,114. We turn
the counts into people per square kilometre so small, dense tracts compare fairly with large ones.
Within the highest-risk tracts, we then take the top quarter for adults 75 and older and the top
quarter for children under 5.

In [ ]:
vuln = pd.DataFrame(client.get_vulnerability_data(PROJECT, limit=5000)["vulnerability"])
ages = vuln[["name", "pop_elderly_75plus", "pop_under5"]]

high = risk[risk["highest_risk"]].merge(ages, on="name")
high["area_km2"] = high.to_crs(6933).area / 1e6
high["older_per_km2"] = high["pop_elderly_75plus"] / high["area_km2"]
high["young_per_km2"] = high["pop_under5"] / high["area_km2"]

high["older"] = high["older_per_km2"] >= high["older_per_km2"].quantile(0.75)
high["young"] = high["young_per_km2"] >= high["young_per_km2"].quantile(0.75)
high["group"] = np.select(
    [high["older"] & high["young"], high["older"], high["young"]],
    ["Both", "Adults 75+", "Children under 5"],
    default="Other highest-risk tracts",
)
print(high["group"].value_counts())
print(high[high["group"] == "Both"].nlargest(5, "hot_nights")[["name", "neighborhood", "hot_nights"]])

GROUP_COLOURS = {"Both": "#7b3294", "Adults 75+": "#d7191c", "Children under 5": "#2c7bb6", "Other highest-risk tracts": "#fdae61"}
ax = risk.plot(color="lightgrey", figsize=(7, 8))
for group, colour in GROUP_COLOURS.items():
    high[high["group"] == group].plot(ax=ax, color=colour, label=group)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLOURS.values()],
          labels=list(GROUP_COLOURS), loc="upper left", bbox_to_anchor=(1, 1))
ax.set_title("Highest-risk tracts by who lives there")
ax.set_axis_off()

Of the 311 highest-risk tracts, 78 are in the top quarter for adults 75 and older, 78 for children
under 5, and 36 for both. Census Tract 116 in Sunset Park, the tract the presentation follows, is one
of the 36 and has the most hot nights among them, 79.

Other age groups work the same way. `pop_elderly_65plus` gives adults 65 and older, and the `frac_`
columns give each group as a share of the tract's population.

## 5. Add your own data

The API covers heat and population. Much of what matters locally lives in other files. As an
example, NYC Aging publishes the location of every older adult center in the city on
[NYC Open Data](https://data.cityofnewyork.us/d/u7wp-np5k). A copy of the 301 centers sits in this
repository so the step works for a full room at once. We count the centers within 400 metres, about
a five-minute walk, of each highest-risk tract with many older residents.

In [ ]:
CENTERS_URL = "https://nishantkishore.com/workshop/files/nyc-older-adult-centers.csv"
centers = pd.read_csv(CENTERS_URL)
centers = gpd.GeoDataFrame(centers, geometry=gpd.points_from_xy(centers["longitude"], centers["latitude"]), crs=4326)

older = high[high["older"]].to_crs(32618)
points = centers.to_crs(32618)
older["centers_400m"] = [int((points.distance(shape) <= 400).sum()) for shape in older.geometry]
print((older["centers_400m"] > 0).sum(), "of", len(older), "have a center within 400 m")
print(older[older["centers_400m"] == 0][["neighborhood", "hot_nights"]].sort_values("hot_nights", ascending=False))

57 of the 78 tracts have an older adult center within 400 metres. The other 21 are a natural first
list for outreach. The same pattern works for any file with coordinates or tract IDs, such
as clinics, cooling centers, schools, or your own survey sites.

## 6. Start your own project

A project starts from a GeoJSON FeatureCollection with one feature per area and a `name` property on
each feature. The name is how every result comes back, so use names people will recognize. Census
tracts, wards, districts, or shapes drawn by hand at [geojson.io](https://geojson.io) all work.

The example below makes a small project from three Sunset Park tracts. Swap in your own file and
project ID.

In [ ]:
import json
import time

my_areas = tracts[tracts["geoid"].isin(["36047011200", "36047011400", "36047011600"])]
my_areas = my_areas.assign(name=my_areas["neighborhood"] + " " + my_areas["geoid"].str[-6:])[["name", "geometry"]]
geojson = json.loads(my_areas.to_json())
# Or read your own file:  geojson = json.load(open("my_areas.geojson"))

MY_PROJECT = "my-first-project"
print(client.create_project(MY_PROJECT, geojson)["message"])

while not client.get_project_status(MY_PROJECT).get("start"):
    time.sleep(30)
print("Ready")

mine = pd.DataFrame(client.iter_metrics(MY_PROJECT))
print(mine[["name", "date", "day_t2m_max", "nighttime_t2m_min"]].tail())

A new project fills in within a few minutes. It starts with the last two weeks of daily heat metrics,
then adds age groups, land cover, and air quality shortly after. Steps 2 to 5 work the same way with
`PROJECT` set to your own ID, your own boundary file in place of the tract file, and a date from your
project's history in step 2. The hot-night count in step 3 grows by one night each day, and
`backfill_project()` adds up to 30 more days of history before the start date. The tract-level adjustment from step 2 is added by the daily update, which
runs at 06:00 UTC.

## 7. Try another city

Every HeatReady key can read the public projects below. Each has a boundary file on the workshop site,
and each feature's `name` matches the name the API returns, so the steps above work on any of them.

| Project | Place | Areas | Boundaries | Shapes from |
|---|---|---|---|---|
| `nyc-manhattan-brooklyn-2026` | Manhattan and Brooklyn | 1,114 census tracts | [GeoJSON](https://nishantkishore.com/workshop/files/projects/nyc-manhattan-brooklyn-2026.geojson) | [US Census TIGER/Line 2025](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) |
| `boston-cambridge-2026` | Boston and Cambridge | 591 census tracts | [GeoJSON](https://nishantkishore.com/workshop/files/projects/boston-cambridge-2026.geojson) | [US Census TIGER/Line 2025](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) |
| `delhi-2026` | Delhi | 446 wards, villages, and towns | [GeoJSON](https://nishantkishore.com/workshop/files/projects/delhi-2026.geojson) | MCD wards (2022) and [geoBoundaries](https://www.geoboundaries.org) |
| `mexico-example` | Mexico City | 1,182 postal codes | [GeoJSON](https://nishantkishore.com/workshop/files/projects/mexico-example.geojson) | Postal-code areas, names from [SEPOMEX](https://github.com/d3249/mexico_zipcodes) |
| `2026-demo-paris-fr` | Paris | 992 IRIS blocks | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-paris-fr.geojson) | [IGN Contours IRIS](https://geoservices.ign.fr/contoursiris) |
| `2026-demo-dhaka-bd` | Bangladesh | 64 districts | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-dhaka-bd.geojson) | [geoBoundaries](https://www.geoboundaries.org) |
| `2026-demo-lagos-ng` | Greater Lagos | 21 local government areas | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-lagos-ng.geojson) | [geoBoundaries](https://www.geoboundaries.org) |
| `2026-demo-oaxaca-mx` | Oaxaca highlands | 50 municipios | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-oaxaca-mx.geojson) | [geoBoundaries](https://www.geoboundaries.org) |
| `2026-demo-phoenix-az` | Arizona and neighbouring counties | 17 counties | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-phoenix-az.geojson) | [geoBoundaries](https://www.geoboundaries.org) |
| `2026-demo-seville-es` | Seville and western Andalusia | 120 municipios | [GeoJSON](https://nishantkishore.com/workshop/files/projects/2026-demo-seville-es.geojson) | [geoBoundaries](https://www.geoboundaries.org) |

The code below maps the latest daily high for one project. Change `PROJECT` to any ID in the table.
The colour scale is fixed from 10 to 40 °C, so the same colour means the same temperature in every
city.

In [ ]:
from datetime import date, timedelta

PROJECT = "mexico-example"
BOUNDARIES_URL = f"https://nishantkishore.com/workshop/files/projects/{PROJECT}.geojson"
areas = gpd.read_file(BOUNDARIES_URL)

since = (date.today() - timedelta(days=7)).isoformat()
week = pd.DataFrame(client.iter_metrics(PROJECT, date_from=since))
latest = week[week["date"] == week["date"].max()]
latest = latest.assign(high=[tract_high(r) for r in latest.to_dict("records")])[["name", "date", "high"]]

city = areas.merge(latest, on="name")
ax = city.plot(column="high", cmap="RdYlBu_r", vmin=10, vmax=40, legend=True, figsize=(7, 7),
               legend_kwds={"label": "Daily high (°C)", "shrink": 0.6})
ax.set_title(f"{PROJECT}, {latest['date'].iloc[0]}")
ax.set_axis_off()
print(len(city), "of", len(areas), "areas mapped")

In Mexico City the daily high in the last week of September ran from about 13 °C in the mountains of the
southwest to about 30 °C in the east of the city.

The project's own areas replace the census tracts here, so a step that uses NYC-only columns, such as
`neighborhood` or `borough`, needs those lines dropped for another city.

A workshop key stops working one week after it is claimed, and its projects are deleted then. To keep
them, make your account permanent at [nishantkishore.com/workshop](https://nishantkishore.com/workshop) before then.
The full list of actions a workshop key can use is in the
[API reference](https://nishantkishore.com/workshop/api).